# Fair Dataset — Multi-Channel CNN for Cortical Layer Classification

## Design decisions (vs `clean_dataset_CNN.ipynb`)

| Decision | Previous notebook | This notebook |
|----------|------------------|---------------|
| Trials per neuron | 1 (first trial only) | Session×layer balanced neuron pool, then all Clip trials for selected neurons |
| Trial sampling | First encountered | Random balanced neuron sampling within each (session, layer) |
| Session normalization | None | **None** (behavior channels kept raw-resampled) |
| Stimulus scope (training) | All stimuli mixed | **Clip only** by default, with optional oracle-only mode |
| Generalization evaluation | None | Trained model evaluated on **Monet2** and **Trippy** (test-set neurons only) |
| Split sanity check | None | **Leave-one-session-out** reported alongside standard split |

## Why session×layer balancing
If each session contributes the same number of neurons per layer, session identity cannot act as a label shortcut.
This directly removes the session-layer leakage pathway in the sampled dataset, without normalizing away biological signal.

## Pipeline overview
1. Load + filter structural metadata (best_only, V1, excitatory, matched)
2. **Phase A+B** — pass through all H5 trials, record trial metadata (neuron, session, trial_idx, hash)
3. Classify hashes by stimulus type
4. **Phase C** — session×layer balanced neuron sampling (optional oracle-only hashes)
5. **Phase D** — build multichan arrays for selected trials (raw behavior channels)
6. Train/test split by `nucleus_id` (StratifiedGroupKFold)
7. Build Monet2 / Trippy generalisation sets (test-set neurons only)
8. Train three CNN architectures on Clip
9. Evaluate: Clip test | Monet2 | Trippy | Leave-one-session-out
10. Per-neuron majority vote + visualisations

In [1]:
%reload_ext autoreload
%autoreload 2

import sys, os, warnings, importlib, subprocess
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    classification_report, ConfusionMatrixDisplay,
)

import microns_datacleaner as mic
import microns_datacleaner.filters as fl

warnings.filterwarnings('ignore')
RNG = np.random.default_rng(42)

# ── Workspace root ────────────────────────────────────────────────────────────
def _find_root():
    p = Path.cwd()
    while p != p.parent:
        if (p / 'microns.h5').exists(): return p
        p = p.parent
    raise RuntimeError('Cannot find workspace root (no microns.h5).')

ROOT      = _find_root()
TOMMY_DIR = (Path.cwd() / '..').resolve()
for d in (ROOT, TOMMY_DIR):
    if str(d) not in sys.path: sys.path.insert(0, str(d))

print(f'Workspace root : {ROOT}')
print(f'Tommy dir      : {TOMMY_DIR}')

Workspace root : c:\Users\tomma\Desktop\NEUROSCIENCE
Tommy dir      : C:\Users\tomma\Desktop\NEUROSCIENCE\neuroscience-project\tommy


## 1. Load structural metadata and apply filters

In [2]:
cleaner = mic.MicronsDataCleaner(
    datadir=os.path.relpath(str(ROOT / 'data')),
    version=1718,
    download_policy='minimum',
)
units_best, _ = cleaner.process_nucleus_data(functional_data='best_only')

v1_only = fl.filter_neurons(units_best, brain_area='V1').copy()
v1_exc  = fl.filter_neurons(v1_only,   cell_type='excitatory_neuron').copy()
matched = fl.filter_neurons(v1_exc,    tuning='matched').copy()

print(f'After V1 filter              : {len(v1_only):,}')
print(f'After excitatory filter      : {len(v1_exc):,}')
print(f'After matched filter         : {len(matched):,}')

unit_id_col    = next(c for c in ['unit_id','functional_unit_id','index'] if c in matched.columns)
neuron_key_col = 'target_id' if 'target_id' in matched.columns else unit_id_col
EXCLUDE_LAYERS = {'L1'}

print(f'\nUnit id column    : {unit_id_col}')
print(f'Neuron key column : {neuron_key_col}')
print('\nLayer counts:')
print(matched['layer'].value_counts().sort_index().to_string())

Transform positions: 100%|██████████| 94014/94014 [00:01<00:00, 88874.41it/s]


After V1 filter              : 59,201
After excitatory filter      : 42,127
After matched filter         : 8,910

Unit id column    : unit_id
Neuron key column : unit_id

Layer counts:
layer
L1        15
L2/3    4247
L4      2670
L5      1615
L6       363


## 2. Open functional H5 and helper functions

In [3]:
from reader import MicronsReader

functional_path = next(
    p for p in [ROOT / 'microns.h5', ROOT / 'data' / 'functional' / 'microns_functional.h5']
    if p.exists()
)
funcreader = MicronsReader(str(functional_path))

scan_rows = []
for k in funcreader.f['sessions'].keys():
    parts = k.split('_')
    if len(parts) == 2:
        try: scan_rows.append((int(parts[0]), int(parts[1]), k))
        except ValueError: pass

scan_df = (
    pd.DataFrame(scan_rows, columns=['session', 'scan_idx', 'key'])
    .sort_values(['session', 'scan_idx']).reset_index(drop=True)
)

matched_scans = matched[['session','scan_idx']].dropna().drop_duplicates().copy()
matched_scans['session']  = matched_scans['session'].astype(int)
matched_scans['scan_idx'] = matched_scans['scan_idx'].astype(int)
scan_candidates = (
    scan_df.merge(matched_scans, on=['session','scan_idx'], how='inner')
    .sort_values(['session','scan_idx']).reset_index(drop=True)
)
print(f'H5 sessions available   : {len(scan_df)}')
print(f'Candidate session pairs : {len(scan_candidates)}')
print(scan_candidates.to_string())

H5 sessions available   : 14
Candidate session pairs : 13
    session  scan_idx  key
0         4         7  4_7
1         5         6  5_6
2         5         7  5_7
3         6         2  6_2
4         6         4  6_4
5         6         6  6_6
6         6         7  6_7
7         7         3  7_3
8         7         5  7_5
9         8         5  8_5
10        9         3  9_3
11        9         4  9_4
12        9         6  9_6


In [4]:
# ── Constants ─────────────────────────────────────────────────────────────────
TARGET_LEN  = 300
MIN_TRACE   = 60
N_CHANNELS  = 12
MAX_SAMPLES = 50_000

# Sampling controls
USE_ORACLE_ONLY = False          # True -> keep only hashes present in every active session
STRICT_GLOBAL_BALANCE = False    # True -> one global min across all (session, layer) with nonzero counts
MIN_SESSIONS_PER_LAYER = 2       # soft mode: keep only layers represented in at least this many sessions

STIM_CHANNEL_NAMES = [
    'stim_mean','stim_std','stim_motion',
    'stim_grad','stim_edge_frac','stim_center_surround','stim_norm_contrast',
]
BEH_CHANNEL_NAMES  = ['treadmill','pupil_x','pupil_y','pupil_size']
CHANNEL_NAMES      = ['neural_response'] + STIM_CHANNEL_NAMES + BEH_CHANNEL_NAMES

# Clip dimensions: H=144,W=256 | Monet2: H=126,W=216 | Trippy: H=90,W=160
HASH_TYPE_MAP = {(144, 256): 'clip', (126, 216): 'monet2', (90, 160): 'trippy'}


# ── Core signal utilities ──────────────────────────────────────────────────────
def zscore_1d(x, eps=1e-8):
    x = np.asarray(x, dtype=np.float32)
    m, s = np.nanmean(x), np.nanstd(x)
    if not np.isfinite(s) or s < eps: return np.zeros_like(x)
    return (x - m) / (s + eps)


def resample_to(arr, target_len):
    arr = np.asarray(arr, dtype=np.float32).ravel()
    if arr.size == target_len: return arr
    return np.interp(
        np.linspace(0, 1, target_len),
        np.linspace(0, 1, arr.size),
        arr,
    ).astype(np.float32)


def extract_stim_channels(clip, target_len):
    """7 per-frame stimulus features from (F,H,W) clip. Returns (7, target_len), z-scored."""
    frames = clip.astype(np.float32)
    if frames.ndim != 3: return np.zeros((7, target_len), dtype=np.float32)
    F, H, W = frames.shape
    eps = 1e-8
    pix           = frames.reshape(F, -1)
    mean_i        = pix.mean(1)
    std_i         = pix.std(1)
    motion        = np.concatenate([[0.], np.abs(np.diff(frames,axis=0)).reshape(F-1,-1).mean(1)]) if F>1 else np.zeros(F)
    grad_e = edge_f = np.zeros(F, dtype=np.float32)
    for i in range(F):
        gy, gx = np.gradient(frames[i]); gm = np.sqrt(gx*gx+gy*gy)
        grad_e[i] = gm.mean(); edge_f[i] = (gm > np.percentile(gm,85)).mean()
    h0,h1,w0,w1 = H//4,(3*H)//4,W//4,(3*W)//4
    ctr   = frames[:,h0:h1,w0:w1].reshape(F,-1)
    cs    = ctr.mean(1) - (pix.sum(1)-ctr.sum(1)) / max((H*W)-((h1-h0)*(w1-w0)),1)
    nc    = std_i / (mean_i + eps)
    raw   = [mean_i, std_i, motion, grad_e, edge_f, cs, nc]
    out   = np.zeros((7, target_len), dtype=np.float32)
    for i, ch in enumerate(raw): out[i] = zscore_1d(resample_to(ch, target_len))
    return out


def extract_behaviour_raw(trial, target_len):
    """Extract + resample behavioural channels without session-wise normalization."""
    result = np.zeros((4, target_len), dtype=np.float32)
    tr = np.asarray(trial['treadmill'], dtype=np.float32).ravel()
    tr = np.where(np.isfinite(tr), tr, 0.)
    result[0] = resample_to(tr, target_len)
    pupil = np.asarray(trial['pupil'], dtype=np.float32)
    if pupil.ndim == 2:
        if pupil.shape[1] == 3 and pupil.shape[0] != 3: pupil = pupil.T
        if pupil.shape[0] >= 3:
            for i in range(3):
                ch = np.where(np.isfinite(pupil[i]), pupil[i], 0.)
                result[1+i] = resample_to(ch, target_len)
    elif pupil.ndim == 1 and pupil.size > 0:
        result[3] = resample_to(np.where(np.isfinite(pupil), pupil, 0.), target_len)
    return result


def get_hash_type(clip):
    """Classify stimulus type from video array dimensions."""
    if clip is None: return 'unknown'
    F, H, W = clip.shape
    return HASH_TYPE_MAP.get((H, W), 'unknown')

## 3. Phase A + B — Trial metadata pass

**One pass through the H5.**  
For every trial in every candidate session we:
- load trial references once
- record `(neuron_key, session_key, trial_idx, hash)` for every matched neuron present

No multichan arrays are built here — this pass is cheap.

In [5]:
trial_meta = []  # list of dicts: one row per (neuron, trial)

for _, srow in scan_candidates.iterrows():
    session  = int(srow['session'])
    scan_idx = int(srow['scan_idx'])
    key      = srow['key']

    sess_neurons = matched[
        (matched['session'] == session) & (matched['scan_idx'] == scan_idx)
    ].dropna(subset=[unit_id_col, 'layer', neuron_key_col]).copy()
    sess_neurons['layer'] = sess_neurons['layer'].astype(str)
    sess_neurons = sess_neurons[~sess_neurons['layer'].isin(EXCLUDE_LAYERS)]
    if sess_neurons.empty: continue

    hashes       = funcreader.get_hashes_by_session(key)
    if not len(hashes): continue

    unit_ids       = funcreader.f[f'sessions/{key}/meta/unit_ids'][:]
    uid_to_row     = {int(u): idx for idx, u in enumerate(unit_ids)}

    # Build neuron lookup for this session
    session_neuron_info = []  # (neuron_key, unit_id, layer, row_in_response)
    for _, nr in sess_neurons.iterrows():
        nkey  = int(nr[neuron_key_col])
        fuid  = int(nr[unit_id_col])
        row   = uid_to_row.get(fuid)
        if row is not None:
            session_neuron_info.append((nkey, fuid, str(nr['layer']), row))

    if not session_neuron_info: continue

    for t, cond_hash in enumerate(hashes):
        try:
            td = funcreader.get_trial(key, t)
        except (ValueError, KeyError):
            continue
        if td is None: continue

        # Record trial metadata for every matched neuron
        for nkey, fuid, layer, row_idx in session_neuron_info:
            if row_idx < td['responses'].shape[0]:
                trial_meta.append({
                    'neuron_key': nkey,
                    'unit_id':    fuid,
                    'layer':      layer,
                    'session_key': key,
                    'session':    session,
                    'scan_idx':   scan_idx,
                    'trial_idx':  t,
                    'hash':       cond_hash,
                    'row_idx':    row_idx,
                })

trial_df = pd.DataFrame(trial_meta)
print(f'Unique sessions processed : {trial_df["session_key"].nunique()}')
print(f'Total (neuron, trial) rows: {len(trial_df):,}')
print(f'Unique neurons            : {trial_df["neuron_key"].nunique():,}')

Unique sessions processed : 13
Total (neuron, trial) rows: 4,127,280
Unique neurons            : 5,267


## 4. Classify hashes by stimulus type

Load each unique hash once, classify by video dimensions.  
The loaded clips are kept in `clip_cache` for reuse in Phase D.

In [6]:
clip_cache     = {}   # hash -> np.ndarray (F,H,W) or None
hash_type_cache = {}  # hash -> 'clip' | 'monet2' | 'trippy' | 'unknown'

unique_hashes = trial_df['hash'].unique()
print(f'Unique hashes to classify: {len(unique_hashes):,}')

for h in unique_hashes:
    video, _ = funcreader.get_video_data(h)
    clip_cache[h]      = video
    hash_type_cache[h] = get_hash_type(video)

trial_df['hash_type'] = trial_df['hash'].map(hash_type_cache)

print('\nTrials by stimulus type:')
print(trial_df['hash_type'].value_counts().to_string())
print('\nTrials by layer x hash_type:')
print(trial_df.groupby(['layer','hash_type']).size().unstack(fill_value=0).to_string())

Unique hashes to classify: 2,247

Trials by stimulus type:
hash_type
clip      3415680
monet2     355800
trippy     355800

Trials by layer x hash_type:
hash_type     clip  monet2  trippy
layer                             
L2/3       1630848  169880  169880
L4         1025280  106800  106800
L5          620160   64600   64600
L6          139392   14520   14520


## 5. Phase C — Session × layer balanced sampling (Clip)

Build a balanced neuron pool so session identity is not predictive of layer:
- start from Clip trials (optionally oracle-only hashes)
- count unique neurons per `(session, layer)`
- sample the same number of neurons per layer across sessions

Default mode is **soft per-layer balancing** (per-layer minimum across sessions where that layer exists).
Set `STRICT_GLOBAL_BALANCE = True` for a single global minimum across all nonzero `(session, layer)` pairs.

In [7]:
candidate_df = trial_df[trial_df['hash_type'] == 'clip'].copy()
print(f'Clip rows before balancing: {len(candidate_df):,}')

# Optional strictest setting: keep only hashes that appear in every active session.
if USE_ORACLE_ONLY:
    n_sessions_clip = candidate_df['session_key'].nunique()
    hash_session_counts = candidate_df.groupby('hash')['session_key'].nunique()
    oracle_hashes = hash_session_counts[hash_session_counts == n_sessions_clip].index
    candidate_df = candidate_df[candidate_df['hash'].isin(oracle_hashes)].copy()
    print(f'Oracle-only mode ON: kept {len(oracle_hashes):,} hashes across {n_sessions_clip} sessions')

if candidate_df.empty:
    raise RuntimeError('No candidate rows after Clip/oracle filtering.')

# Work at the neuron level for balancing; trials are expanded after neuron selection.
neuron_pool = candidate_df[['session_key', 'layer', 'neuron_key']].drop_duplicates().copy()
pair_counts = (
    neuron_pool.groupby(['layer', 'session_key'])['neuron_key']
    .nunique()
    .unstack(fill_value=0)
    .sort_index()
)

print('\nUnique neurons per (layer, session):')
print(pair_counts.to_string())

selected_neuron_rows = []

if STRICT_GLOBAL_BALANCE:
    nonzero = pair_counts[pair_counts > 0].stack()
    if nonzero.empty:
        raise RuntimeError('No nonzero (layer, session) pairs available for strict balancing.')
    global_min = int(nonzero.min())
    print(f'\nStrict global balancing ON: sampling {global_min} neurons per nonzero (layer, session) pair')

    for layer in pair_counts.index:
        for s_key, cnt in pair_counts.loc[layer].items():
            if cnt <= 0:
                continue
            grp = neuron_pool[(neuron_pool['layer'] == layer) & (neuron_pool['session_key'] == s_key)]
            picked = RNG.choice(grp['neuron_key'].to_numpy(), size=global_min, replace=False)
            selected_neuron_rows.extend(
                {'session_key': s_key, 'layer': layer, 'neuron_key': int(nk)} for nk in picked
            )
else:
    print('\nSoft per-layer balancing ON')
    for layer in pair_counts.index:
        row = pair_counts.loc[layer]
        present = row[row > 0]
        if len(present) < MIN_SESSIONS_PER_LAYER:
            print(f'  skip layer {layer}: only {len(present)} represented session(s)')
            continue

        layer_min = int(present.min())
        print(f'  layer {layer}: {len(present)} sessions, sample {layer_min} neurons/session')

        for s_key in present.index:
            grp = neuron_pool[(neuron_pool['layer'] == layer) & (neuron_pool['session_key'] == s_key)]
            picked = RNG.choice(grp['neuron_key'].to_numpy(), size=layer_min, replace=False)
            selected_neuron_rows.extend(
                {'session_key': s_key, 'layer': layer, 'neuron_key': int(nk)} for nk in picked
            )

selected_neurons_df = pd.DataFrame(selected_neuron_rows)
if selected_neurons_df.empty:
    raise RuntimeError('Balanced sampler selected zero neurons. Relax balancing settings.')

# Expand selected neuron pool back to trial rows.
selected_df = candidate_df.merge(
    selected_neurons_df,
    on=['session_key', 'layer', 'neuron_key'],
    how='inner'
)

print(f'\nSelected neurons           : {selected_neurons_df["neuron_key"].nunique():,}')
print(f'Selected sample rows       : {len(selected_df):,}')
print('\nSelected neurons by layer:')
print(selected_neurons_df['layer'].value_counts().sort_index().to_string())

balanced_counts = (
    selected_neurons_df.groupby(['layer', 'session_key'])['neuron_key']
    .nunique()
    .unstack(fill_value=0)
    .sort_index()
)
print('\nBalanced neuron counts per (layer, session):')
print(balanced_counts.to_string())

Clip rows before balancing: 3,415,680

Unique neurons per (layer, session):
session_key  4_7  5_6  5_7  6_2  6_4  6_6  6_7  7_3  7_5  8_5  9_3  9_4  9_6
layer                                                                       
L2/3         364  151  184  279  497  423  462  203  176  400    0  680  428
L4           247  249  285  294  143   29   66  190   77  272  818    0    0
L5           104  189  147  157  213  198  319  159   21  108    0    0    0
L6             0   97   85   44   63   47   27    0    0    0    0    0    0

Soft per-layer balancing ON
  layer L2/3: 12 sessions, sample 151 neurons/session
  layer L4: 11 sessions, sample 29 neurons/session
  layer L5: 10 sessions, sample 21 neurons/session
  layer L6: 6 sessions, sample 27 neurons/session

Selected neurons           : 2,033
Selected sample rows       : 961,152

Selected neurons by layer:
layer
L2/3    1812
L4       319
L5       210
L6       162

Balanced neuron counts per (layer, session):
session_key  4_7  5_6 

## 6. Phase D — Build multichannel arrays

For each selected `(neuron, trial)`:
- **Neural** (ch 0): raw trace → per-trace z-score
- **Stimulus** (ch 1–7): from clip video, z-scored per channel per trial
- **Behavioural** (ch 8–11): raw-resampled channels (no session normalization)

Session leakage control is handled in sampling (Phase C), not by channel normalization.

In [8]:
records = []  # list of dicts: neuron_key, layer, multichan, session_key

# Group by (session, trial_idx) to avoid redundant H5 reads
grouped = selected_df.groupby(['session_key', 'trial_idx'])
n_groups = len(grouped)
print(f'Unique (session, trial) pairs to load: {n_groups:,}')

skipped = 0
for g_idx, ((s_key, t_idx), grp) in enumerate(grouped):
    if g_idx % 1000 == 0:
        print(f'  {g_idx:,}/{n_groups:,}  records so far: {len(records):,}', end='\r')

    try:
        td = funcreader.get_trial(s_key, t_idx)
    except (ValueError, KeyError):
        skipped += 1; continue
    if td is None:
        skipped += 1; continue

    responses = td['responses']   # (n_units, n_frames)

    # Behavioural channels: raw-resampled, no session normalization
    beh_raw = extract_behaviour_raw(td, TARGET_LEN)       # (4, T)

    # Stimulus channels (cached)
    cond_hash = grp['hash'].iloc[0]
    clip_arr  = clip_cache.get(cond_hash)
    stim_ch   = extract_stim_channels(clip_arr, TARGET_LEN) if clip_arr is not None \
                else np.zeros((7, TARGET_LEN), dtype=np.float32)

    # Per-neuron neural channel
    for _, row in grp.iterrows():
        row_idx = int(row['row_idx'])
        if row_idx >= responses.shape[0]: continue

        trace = np.asarray(responses[row_idx, :], dtype=np.float32).ravel()
        trace = trace[np.isfinite(trace)]
        if trace.size < MIN_TRACE:
            skipped += 1; continue

        multichan         = np.zeros((N_CHANNELS, TARGET_LEN), dtype=np.float32)
        multichan[0]      = zscore_1d(resample_to(trace, TARGET_LEN))
        multichan[1:8]    = stim_ch
        multichan[8:12]   = beh_raw

        records.append({
            'neuron_key':  int(row['neuron_key']),
            'layer':       row['layer'],
            'session_key': s_key,
            'multichan':   multichan,
        })

print(f'\nBuilt {len(records):,} samples  |  skipped {skipped}')
print('Samples by layer:')
print(pd.Series([r['layer'] for r in records]).value_counts().sort_index().to_string())

Unique (session, trial) pairs to load: 4,992
  4,000/4,992  records so far: 838,688
Built 961,152 samples  |  skipped 0
Samples by layer:
L2/3    695808
L4      122496
L5       80640
L6       62208


In [9]:
# save records to disk for later use
import pickle
with open(TOMMY_DIR / 'clip_records.pkl', 'wb') as f:
    pickle.dump(records, f)


## 7. Train / test split by neuron identity

Split by `neuron_key` so all trials of the same neuron land on the same side.  
Stratified on layer to preserve class proportions.  
Rare classes with fewer than 20 unique neurons are dropped.

In [10]:
records_df = pd.DataFrame([{'neuron_key': r['neuron_key'], 'layer': r['layer']} for r in records])
neuron_df  = records_df.drop_duplicates(subset='neuron_key')

layer_counts = neuron_df['layer'].value_counts()
keep_layers  = layer_counts[layer_counts >= 20].index.tolist()
neuron_df    = neuron_df[neuron_df['layer'].isin(keep_layers)].copy()

print('Unique neurons per layer (after rare-class filter):')
print(neuron_df['layer'].value_counts().sort_index().to_string())

enc = LabelEncoder()
enc.fit(neuron_df['layer'])
n_classes = len(enc.classes_)
print(f'\nClasses: {list(enc.classes_)}')

# 75/25 stratified split by neuron (not by trial)
train_neurons, test_neurons = train_test_split(
    neuron_df['neuron_key'].values,
    test_size=0.25,
    random_state=42,
    stratify=neuron_df['layer'].values,
)
train_set = set(train_neurons)
test_set  = set(test_neurons)
print(f'\nTrain neurons: {len(train_set):,} | Test neurons: {len(test_set):,}')


def build_split(neuron_keys_set):
    X, y, uids = [], [], []
    for r in records:
        if r['neuron_key'] not in neuron_keys_set: continue
        if r['layer'] not in keep_layers: continue
        X.append(r['multichan'])
        y.append(r['layer'])
        uids.append(r['neuron_key'])
    return np.asarray(X, dtype=np.float32), enc.transform(y), np.array(uids)


X_train, y_train, uid_train = build_split(train_set)
X_test,  y_test,  uid_test  = build_split(test_set)

print(f'\nTrain shape : {X_train.shape}')
print(f'Test shape  : {X_test.shape}')
print('\nTrain class distribution:')
print(pd.Series(enc.inverse_transform(y_train)).value_counts().sort_index().to_string())
print('\nTest class distribution:')
print(pd.Series(enc.inverse_transform(y_test)).value_counts().sort_index().to_string())

Unique neurons per layer (after rare-class filter):
layer
L2/3    1391
L4       296
L5       194
L6       152

Classes: ['L2/3', 'L4', 'L5', 'L6']

Train neurons: 1,524 | Test neurons: 509

Train shape : (723456, 12, 300)
Test shape  : (237696, 12, 300)

Train class distribution:
L2/3    523776
L4       92544
L5       60288
L6       46848

Test class distribution:
L2/3    172032
L4       29952
L5       20352
L6       15360


## 8. Generalisation datasets — Monet2 and Trippy

Collect all Monet2 and Trippy trials for **test-set neurons only**.  
The model is never retrained; these sets are used only for evaluation.

In [11]:
def build_gen_set(stim_type):
    """Build X, y, uids for a stimulus type, restricted to test-set neurons."""
    gen_df = trial_df[
        (trial_df['hash_type'] == stim_type) &
        (trial_df['neuron_key'].isin(test_set)) &
        (trial_df['layer'].isin(keep_layers))
    ].copy()

    if gen_df.empty:
        print(f'No {stim_type} trials for test-set neurons.')
        return None, None, None

    X_gen, y_gen, uid_gen = [], [], []
    skipped = 0
    for (s_key, t_idx), grp in gen_df.groupby(['session_key', 'trial_idx']):
        try:
            td = funcreader.get_trial(s_key, t_idx)
        except (ValueError, KeyError):
            skipped += 1; continue
        if td is None:
            skipped += 1; continue

        responses = td['responses']
        beh_raw   = extract_behaviour_raw(td, TARGET_LEN)
        cond_hash = grp['hash'].iloc[0]
        clip_arr  = clip_cache.get(cond_hash)
        stim_ch   = extract_stim_channels(clip_arr, TARGET_LEN) if clip_arr is not None \
                    else np.zeros((7, TARGET_LEN), dtype=np.float32)

        for _, row in grp.iterrows():
            row_idx = int(row['row_idx'])
            if row_idx >= responses.shape[0]: continue
            trace = np.asarray(responses[row_idx, :], dtype=np.float32).ravel()
            trace = trace[np.isfinite(trace)]
            if trace.size < MIN_TRACE:
                skipped += 1; continue
            mc = np.zeros((N_CHANNELS, TARGET_LEN), dtype=np.float32)
            mc[0]    = zscore_1d(resample_to(trace, TARGET_LEN))
            mc[1:8]  = stim_ch
            mc[8:12] = beh_raw
            X_gen.append(mc)
            y_gen.append(row['layer'])
            uid_gen.append(int(row['neuron_key']))

    if not X_gen:
        print(f'No valid {stim_type} samples.')
        return None, None, None

    X_g = np.asarray(X_gen, dtype=np.float32)
    y_g = enc.transform(y_gen)
    print(f'{stim_type.upper():8s}  samples: {len(X_g):,}  neurons: {len(set(uid_gen)):,}')
    print('  by layer:', pd.Series(enc.inverse_transform(y_g)).value_counts().sort_index().to_dict())
    if skipped: print(f'  skipped : {skipped}')
    return X_g, y_g, np.array(uid_gen)


X_monet2, y_monet2, uid_monet2 = build_gen_set('monet2')
X_trippy, y_trippy, uid_trippy = build_gen_set('trippy')

MONET2    samples: 42,880  neurons: 509
  by layer: {'L2/3': 26600, 'L4': 9320, 'L5': 5040, 'L6': 1920}
TRIPPY    samples: 42,880  neurons: 509
  by layer: {'L2/3': 26600, 'L4': 9320, 'L5': 5040, 'L6': 1920}


## 9. PyTorch setup

In [12]:
if importlib.util.find_spec('torch') is None:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--quiet', 'torch',
        '--index-url', 'https://download.pytorch.org/whl/cpu',
    ])

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | device: {DEVICE}')

class_counts  = np.bincount(y_train, minlength=n_classes).astype(np.float32)
class_weights = torch.tensor(1. / (class_counts + 1e-8)).to(DEVICE)
class_weights /= class_weights.sum()
print('Class weights:', dict(zip(enc.classes_, class_weights.cpu().numpy().round(4))))

BATCH_SIZE = 128


def make_loader(X, y, shuffle):
    return DataLoader(
        TensorDataset(torch.tensor(X, dtype=torch.float32),
                      torch.tensor(y, dtype=torch.long)),
        batch_size=BATCH_SIZE, shuffle=shuffle, drop_last=False,
    )


train_loader  = make_loader(X_train,  y_train,  shuffle=True)
test_loader   = make_loader(X_test,   y_test,   shuffle=False)
monet2_loader = make_loader(X_monet2, y_monet2, shuffle=False) if X_monet2 is not None else None
trippy_loader = make_loader(X_trippy, y_trippy, shuffle=False) if X_trippy is not None else None

print(f'Train batches: {len(train_loader)} | Test batches: {len(test_loader)}')

PyTorch 2.11.0+cpu | device: cpu
Class weights: {'L2/3': np.float32(0.0377), 'L4': np.float32(0.2134), 'L5': np.float32(0.3275), 'L6': np.float32(0.4215)}
Train batches: 5652 | Test batches: 1857


## 10. CNN architectures

Three architectures — same as `clean_dataset_CNN.ipynb`.

| Model | Description |
|-------|-------------|
| **ShallowCNN** | 3 Conv blocks → MLP head |
| **ResidualCNN** | Residual blocks with adaptive pooling |
| **MultiStreamCNN** | Parallel first-layer branches (neural+beh, neural+stim, all channels) |

In [13]:
class MultiChannelShallowCNN(nn.Module):
    def __init__(self, in_ch, seq_len, n_cls, dropout=0.4):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(in_ch, 64,  7, padding=3), nn.BatchNorm1d(64),  nn.ReLU(True), nn.MaxPool1d(2),
            nn.Conv1d(64,   128,  5, padding=2), nn.BatchNorm1d(128), nn.ReLU(True), nn.MaxPool1d(2),
            nn.Conv1d(128,  256,  3, padding=1), nn.BatchNorm1d(256), nn.ReLU(True), nn.MaxPool1d(2),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256*(seq_len//8), 256), nn.ReLU(True), nn.Dropout(dropout),
            nn.Linear(256, n_cls),
        )
    def forward(self, x): return self.head(self.conv(x))


class ResBlock1d(nn.Module):
    def __init__(self, ch, k=3):
        super().__init__()
        p = k//2
        self.net = nn.Sequential(
            nn.Conv1d(ch,ch,k,padding=p,bias=False), nn.BatchNorm1d(ch), nn.ReLU(True),
            nn.Conv1d(ch,ch,k,padding=p,bias=False), nn.BatchNorm1d(ch),
        )
        self.act = nn.ReLU(True)
    def forward(self, x): return self.act(self.net(x)+x)


class MultiChannelResidualCNN(nn.Module):
    def __init__(self, in_ch, seq_len, n_cls, dropout=0.4):
        super().__init__()
        self.stem   = nn.Sequential(nn.Conv1d(in_ch,64,7,padding=3,bias=False),nn.BatchNorm1d(64),nn.ReLU(True))
        self.stage1 = nn.Sequential(ResBlock1d(64), nn.Conv1d(64,128,3,stride=2,padding=1,bias=False),nn.BatchNorm1d(128),nn.ReLU(True))
        self.stage2 = nn.Sequential(ResBlock1d(128),nn.Conv1d(128,256,3,stride=2,padding=1,bias=False),nn.BatchNorm1d(256),nn.ReLU(True))
        self.stage3 = nn.Sequential(ResBlock1d(256))
        self.pool   = nn.AdaptiveAvgPool1d(1)
        self.head   = nn.Sequential(
            nn.Flatten(), nn.Dropout(dropout),
            nn.Linear(256,128), nn.ReLU(True), nn.Dropout(dropout),
            nn.Linear(128, n_cls),
        )
    def forward(self, x):
        return self.head(self.pool(self.stage3(self.stage2(self.stage1(self.stem(x))))))


class MultiStreamCNN(nn.Module):
    NB_IDX  = [0, 8, 9, 10, 11]
    NS_IDX  = list(range(8))
    ALL_IDX = list(range(12))

    def __init__(self, n_ch, seq_len, n_cls, k_nb=48, k_ns=48, k_all=32, dropout=0.4):
        super().__init__()
        def _branch(ic,oc): return nn.Sequential(
            nn.Conv1d(ic,oc,7,padding=3,bias=False), nn.BatchNorm1d(oc), nn.ReLU(True), nn.MaxPool1d(2),
        )
        self.branch_nb  = _branch(len(self.NB_IDX),  k_nb)
        self.branch_ns  = _branch(len(self.NS_IDX),  k_ns)
        self.branch_all = _branch(len(self.ALL_IDX), k_all)
        fused = k_nb+k_ns+k_all
        self.backbone = nn.Sequential(
            nn.Conv1d(fused,128,5,padding=2,bias=False), nn.BatchNorm1d(128), nn.ReLU(True), nn.MaxPool1d(2),
            nn.Conv1d(128,  256,3,padding=1,bias=False), nn.BatchNorm1d(256), nn.ReLU(True), nn.MaxPool1d(2),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256*(seq_len//8), 256), nn.ReLU(True), nn.Dropout(dropout),
            nn.Linear(256, n_cls),
        )
    def forward(self, x):
        return self.head(self.backbone(torch.cat([
            self.branch_nb(x[:,self.NB_IDX,:]),
            self.branch_ns(x[:,self.NS_IDX,:]),
            self.branch_all(x[:,self.ALL_IDX,:]),
        ], dim=1)))


def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

shallow_cnn  = MultiChannelShallowCNN(N_CHANNELS, TARGET_LEN, n_classes).to(DEVICE)
residual_cnn = MultiChannelResidualCNN(N_CHANNELS, TARGET_LEN, n_classes).to(DEVICE)
multi_stream = MultiStreamCNN(N_CHANNELS, TARGET_LEN, n_classes).to(DEVICE)

print(f'ShallowCNN  params: {count_params(shallow_cnn):,}')
print(f'ResidualCNN params: {count_params(residual_cnn):,}')
print(f'MultiStream params: {count_params(multi_stream):,}')

ShallowCNN  params: 2,572,100
ResidualCNN params: 680,452
MultiStream params: 2,614,420


## 11. Training utilities

In [14]:
def train_model(model, train_loader, val_loader, *,
                lr=3e-4, weight_decay=1e-4,
                max_epochs=150, patience=25, label='model'):
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    opt       = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched     = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_epochs)

    hist = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    best_val, best_state, patience_cnt = float('inf'), None, 0

    for epoch in range(1, max_epochs+1):
        model.train()
        running = 0.
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            opt.step()
            running += loss.item() * xb.size(0)
        train_loss = running / len(train_loader.dataset)

        model.eval()
        val_loss, correct = 0., 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                logits   = model(xb)
                val_loss += criterion(logits, yb).item() * xb.size(0)
                correct  += (logits.argmax(1) == yb).sum().item()
        val_loss /= len(val_loader.dataset)
        val_acc   = correct / len(val_loader.dataset)
        sched.step()

        hist['train_loss'].append(train_loss)
        hist['val_loss'].append(val_loss)
        hist['val_acc'].append(val_acc)

        if val_loss < best_val:
            best_val, best_state, patience_cnt = val_loss, \
                {k: v.cpu().clone() for k,v in model.state_dict().items()}, 0
        else:
            patience_cnt += 1
            if patience_cnt >= patience:
                print(f'[{label}] Early stop at epoch {epoch}')
                break

        if epoch % 1 == 0 or epoch == 1:
            print(f'[{label}] Epoch {epoch:3d} | '
                  f'train={train_loss:.4f}  val={val_loss:.4f}  acc={val_acc:.3f}')

    model.load_state_dict({k: v.to(DEVICE) for k,v in best_state.items()})
    print(f'[{label}] Best checkpoint restored (val_loss={best_val:.4f})')
    return hist


def get_predictions(model, loader):
    model.eval()
    preds = []
    with torch.no_grad():
        for xb, _ in loader:
            preds.append(model(xb.to(DEVICE)).argmax(1).cpu().numpy())
    return np.concatenate(preds)


print('Training utilities defined.')

Training utilities defined.


## 12. Train — all three architectures on Clip

In [ ]:
shallow_cnn  = MultiChannelShallowCNN(N_CHANNELS, TARGET_LEN, n_classes).to(DEVICE)
history_sh   = train_model(shallow_cnn,  train_loader, test_loader, label='ShallowCNN')

In [ ]:
residual_cnn = MultiChannelResidualCNN(N_CHANNELS, TARGET_LEN, n_classes).to(DEVICE)
history_res  = train_model(residual_cnn, train_loader, test_loader, label='ResidualCNN')

In [ ]:
multi_stream = MultiStreamCNN(N_CHANNELS, TARGET_LEN, n_classes).to(DEVICE)
history_ms   = train_model(multi_stream, train_loader, test_loader, label='MultiStreamCNN')

## 13. Training curves

In [ ]:
histories = [
    (history_sh,  'ShallowCNN'),
    (history_res, 'ResidualCNN'),
    (history_ms,  'MultiStreamCNN'),
]

fig, axes = plt.subplots(len(histories), 2, figsize=(14, 4*len(histories)))
for row, (hist, name) in enumerate(histories):
    axes[row,0].plot(hist['train_loss'], label='train')
    axes[row,0].plot(hist['val_loss'],   label='val')
    axes[row,0].set_title(f'{name} — loss'); axes[row,0].legend()
    axes[row,0].set_xlabel('Epoch'); axes[row,0].set_ylabel('Cross-entropy')

    axes[row,1].plot(hist['val_acc'], color='steelblue')
    axes[row,1].axhline(1/n_classes, color='grey', linestyle='--',
                        label=f'chance ({1/n_classes:.2f})')
    axes[row,1].set_title(f'{name} — val accuracy')
    axes[row,1].set_xlabel('Epoch'); axes[row,1].set_ylabel('Accuracy')
    axes[row,1].legend()

plt.tight_layout()
plt.show()

## 14. Evaluation — Clip test | Monet2 | Trippy

The model is trained on Clip only.  
Monet2 and Trippy measure **out-of-domain generalisation**: does the model decode layer  
identity from a synthetic stimulus it has never seen during training?

In [ ]:
model_pairs = [
    (shallow_cnn,  'ShallowCNN'),
    (residual_cnn, 'ResidualCNN'),
    (multi_stream, 'MultiStreamCNN'),
]

eval_domains = [
    ('Clip (test)',   test_loader,   y_test),
    ('Monet2 (gen)',  monet2_loader, y_monet2),
    ('Trippy (gen)',  trippy_loader, y_trippy),
]

rows = []
for model, mname in model_pairs:
    for domain, loader, y_true in eval_domains:
        if loader is None or y_true is None: continue
        y_pred = get_predictions(model, loader)
        rows.append({
            'model':             mname,
            'domain':            domain,
            'accuracy':          accuracy_score(y_true, y_pred),
            'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
            'macro_f1':          f1_score(y_true, y_pred, average='macro'),
        })

eval_df = pd.DataFrame(rows)
print('── Per-domain evaluation ──')
display(eval_df.round(4))

# Detailed report per model × domain
for model, mname in model_pairs:
    for domain, loader, y_true in eval_domains:
        if loader is None or y_true is None: continue
        y_pred = get_predictions(model, loader)
        print(f'\n── {mname} | {domain} ──')
        print(classification_report(y_true, y_pred, target_names=enc.classes_))

## 15. Confusion matrices

In [ ]:
active_domains = [(d, ld, yt) for d, ld, yt in eval_domains if ld is not None and yt is not None]

fig, axes = plt.subplots(
    len(model_pairs), len(active_domains),
    figsize=(6*len(active_domains), 5*len(model_pairs))
)
if len(model_pairs) == 1: axes = axes[np.newaxis, :]
if len(active_domains) == 1: axes = axes[:, np.newaxis]

for r, (model, mname) in enumerate(model_pairs):
    for c, (domain, loader, y_true) in enumerate(active_domains):
        y_pred = get_predictions(model, loader)
        ConfusionMatrixDisplay.from_predictions(
            y_true, y_pred,
            display_labels=enc.classes_,
            normalize='true', cmap='Blues', colorbar=False,
            xticks_rotation=45, ax=axes[r, c],
        )
        axes[r, c].set_title(f'{mname}\n{domain}')

plt.tight_layout()
plt.show()

## 16. Per-neuron majority vote

Aggregate trial-level predictions per `neuron_key` by majority vote.  
Reports both trial-level and neuron-level metrics; the gap between the two  
shows how consistent the model is across trials of the same neuron.

In [ ]:
from scipy.stats import mode as scipy_mode


def majority_vote(model, X, y, uids):
    loader = make_loader(X, y, shuffle=False)
    preds  = get_predictions(model, loader)

    buckets_pred, buckets_true = defaultdict(list), {}
    for uid, pred, true in zip(uids, preds, y):
        buckets_pred[uid].append(pred)
        buckets_true[uid] = true

    voted_pred, voted_true = [], []
    for uid, ps in buckets_pred.items():
        voted_pred.append(scipy_mode(ps, keepdims=False).mode)
        voted_true.append(buckets_true[uid])

    return np.array(voted_true), np.array(voted_pred)


print('Per-neuron majority vote (Clip test set):')
for model, mname in model_pairs:
    vt, vp = majority_vote(model, X_test, y_test, uid_test)
    print(f'\n{mname}  ({len(vt)} neurons)')
    print(f'  Trial-level  balanced acc : {balanced_accuracy_score(y_test, get_predictions(model, test_loader)):.3f}')
    print(f'  Neuron-level balanced acc : {balanced_accuracy_score(vt, vp):.3f}')
    print(f'  Neuron-level macro F1     : {f1_score(vt, vp, average="macro"):.3f}')

## 17. Leave-one-session-out sanity check

Train on all neurons from 13 sessions, evaluate on the held-out session.  
If this score drops sharply vs the standard split, session artefacts are leaking through  
the session normalisation and further work is needed.

In [ ]:
all_session_keys = selected_df['session_key'].unique().tolist()
print(f'Running LOO over {len(all_session_keys)} sessions...')

loo_rows = []
for held_out in all_session_keys:
    train_keys = [k for k in all_session_keys if k != held_out]

    # Build splits from the already-constructed records
    def build_loo(keys_set):
        X_, y_, u_ = [], [], []
        for r in records:
            if r['session_key'] not in keys_set: continue
            if r['layer'] not in keep_layers: continue
            X_.append(r['multichan']); y_.append(r['layer']); u_.append(r['neuron_key'])
        if not X_: return None, None, None
        return np.asarray(X_, dtype=np.float32), enc.transform(y_), np.array(u_)

    Xtr, ytr, _ = build_loo(set(train_keys))
    Xte, yte, _ = build_loo({held_out})
    if Xtr is None or Xte is None or len(np.unique(ytr)) < 2:
        print(f'  {held_out}: skip (too few classes)')
        continue

    loo_train_loader = make_loader(Xtr, ytr, shuffle=True)
    loo_test_loader  = make_loader(Xte, yte, shuffle=False)

    # Use ResidualCNN (most parameter-efficient) for LOO to keep runtime manageable
    loo_model = MultiChannelResidualCNN(N_CHANNELS, TARGET_LEN, n_classes).to(DEVICE)
    train_model(loo_model, loo_train_loader, loo_test_loader,
                max_epochs=80, patience=15, label=f'LOO-{held_out}')
    yp = get_predictions(loo_model, loo_test_loader)
    bacc = balanced_accuracy_score(yte, yp)
    loo_rows.append({'held_out': held_out, 'n_test': len(yte), 'balanced_acc': bacc})
    print(f'  {held_out:6s}  n_test={len(yte):4d}  balanced_acc={bacc:.3f}')

loo_df = pd.DataFrame(loo_rows)
print(f'\nLOO mean balanced_acc : {loo_df["balanced_acc"].mean():.3f} ± {loo_df["balanced_acc"].std():.3f}')
print(f'Standard split (ResidualCNN Clip test) for comparison ↑')

## 18. LOO results vs standard split

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(loo_df['held_out'], loo_df['balanced_acc'], color='steelblue', edgecolor='white')

# Reference: standard split ResidualCNN balanced accuracy
ref_rows = eval_df[(eval_df['model']=='ResidualCNN') & (eval_df['domain']=='Clip (test)')]
if not ref_rows.empty:
    ref = ref_rows['balanced_accuracy'].values[0]
    ax.axhline(ref, color='firebrick', linestyle='--', label=f'Standard split ({ref:.3f})')

ax.axhline(1/n_classes, color='grey', linestyle=':', label=f'Chance ({1/n_classes:.2f})')
ax.set_xlabel('Held-out session')
ax.set_ylabel('Balanced accuracy')
ax.set_title('Leave-one-session-out — ResidualCNN')
ax.legend()
plt.tight_layout()
plt.show()

## Summary

| Step | Detail |
|------|--------|
| Population | best_only + V1 + excitatory + matched → ~8.9k neurons |
| Sampling | Session×layer balanced neuron selection (soft per-layer minimum by default) |
| Optional strictness | `USE_ORACLE_ONLY=True` keeps only hashes shared across all active sessions |
| Behaviour channels | Raw-resampled per trial (no session normalization) |
| Input tensor | (N, 12, 300) — 1 neural + 7 stimulus + 4 behaviour |
| Split | 75% train / 25% test, stratified by layer, split by `neuron_key` |
| Training domain | Clip (natural movies) only |
| Primary metric | Balanced accuracy (class imbalance L2/3 >> L6) |
| Generalisation | Monet2 + Trippy (test-set neurons, zero retraining) |
| Sanity check | Leave-one-session-out (detects residual session artefacts) |

### Reading the results
- **Clip test ≈ LOO**: session leakage is likely well controlled by sampling.
- **Clip test >> LOO**: residual session artefacts likely remain; turn on stricter sampling (`STRICT_GLOBAL_BALANCE` and/or `USE_ORACLE_ONLY`).
- **Monet2/Trippy ≈ Clip test**: model learned layer identity, not clip-specific patterns.
- **Monet2/Trippy << Clip test**: features are stimulus-family-specific; consider retraining on mixed stimuli or analysing which layers fail to generalise.